In [ ]:
import os
import re

root = r'c:\xampp\htdocs\BOLSADETRABAJOFORMULARIO1'
conflict_pattern = re.compile(
    r'<<<<<<< HEAD\r?\n(.*?)\r?\n=======\r?\n.*?\r?\n>>>>>>> .*?\r?\n',
    re.S,
)

changed_files = []
for dirpath, dirnames, filenames in os.walk(root):
    if any(skip in dirpath.split(os.sep) for skip in ['.git', 'vendor', 'node_modules']):
        continue
    for filename in filenames:
        path = os.path.join(dirpath, filename)
        if not path.endswith((
            '.php', '.blade.php', '.env', '.env.example', '.json', '.md', '.yml',
            '.yaml', '.txt', '.js', '.ts', '.ps1', '.lock', '.sql', '.sh'
        )):
            continue
        with open(path, 'r', encoding='utf-8', errors='ignore') as f:
            text = f.read()
        if '<<<<<<< HEAD' in text and '=======' in text and '>>>>>>>' in text:
            original = text
            while True:
                resolved = conflict_pattern.sub(lambda m: m.group(1) + '\n', text)
                if resolved == text:
                    break
                text = resolved
            if text != original:
                with open(path, 'w', encoding='utf-8', newline='\n') as f:
                    f.write(text)
                changed_files.append(os.path.relpath(path, root))

print('Cleaned', len(changed_files), 'files:')
for f in changed_files:
    print(f)


# Resolve Merge Conflicts